# Подготовка данных для прогнозирования

В работе используется уже обработанный файл `data/data_clean.parquet`, полученный в результате обработки данных в прошлых дз (прикреплен вместе в дз).

Логика работы:
1. загружаем и проверяем данные;
2. обрабатываем категориальные признаки;
3. анализируем числовые признаки;
4. создаём дополнительные признаки;
5. проверяем корреляции;
6. обучаем модель на 2022 году и проверяем на 2023 году.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Загрузка обработанных данных


In [2]:
DATA_PATH = Path('../data') / 'data_clean.parquet'


df_pedagog_all = pd.read_parquet(DATA_PATH)
print("Размер данных:", df_pedagog_all.shape)

Размер данных: (417149, 31)


In [3]:
print("Первые строки:")
display(df_pedagog_all.head())
print("\nЦелевая переменная target:")
print(df_pedagog_all['target'].value_counts(dropna=False))

Первые строки:


,id,school_num,main_position,speciality,sex,age,academic_degree,category_qual,category_date,category_due,...,n_unique_subjects,min_parallel,max_parallel,n_unique_parallel,target,experience_age_all_clean,experience_age_pedagogic_clean,experience_age_other_clean,category_date_clean,category_due_clean
0,f583c34a49,478,Педагог дополнительного образования,не указано,0,35,нет / не указано,нет категории,<NA>,<NA>,...,2,2,9,3,0.0,12,1,11,<NA>,<NA>
1,f583c34a49,478,Педагог дополнительного образования,не указано,0,35,нет / не указано,нет категории,<NA>,<NA>,...,2,2,9,3,0.0,12,1,11,<NA>,<NA>
2,f583c34a49,478,Педагог дополнительного образования,не указано,0,35,нет / не указано,нет категории,<NA>,<NA>,...,2,2,9,3,0.0,12,1,11,<NA>,<NA>
3,f583c34a49,478,Педагог дополнительного образования,не указано,0,35,нет / не указано,нет категории,<NA>,<NA>,...,2,2,9,3,0.0,12,1,11,<NA>,<NA>
4,08ff701e26,523,Учитель,не указано,0,54,нет / не указано,нет категории,<NA>,<NA>,...,2,7,11,5,1.0,29,24,5,<NA>,<NA>



Целевая переменная target:
target
0.0    234488
NaN    149690
1.0     32971
Name: count, dtype: int64


## 2. Целевая переменная и очищенные признаки

In [4]:
# Целевая переменная и очищенные числовые признаки
df_pedagog_all['is_out_next_year'] = df_pedagog_all['target']
df_pedagog_all = df_pedagog_all.drop(columns=['target'])

clean_cols = {
    'experience_age_all_clean': 'experience_age_all',
    'experience_age_pedagogic_clean': 'experience_age_pedagogic',
    'experience_age_other_clean': 'experience_age_other',
    'category_date_clean': 'category_date',
    'category_due_clean': 'category_due',
}

for clean_col, col in clean_cols.items():
    if clean_col in df_pedagog_all.columns:
        df_pedagog_all[col] = df_pedagog_all[clean_col]